In [0]:
   # Project configuration

catalog_name = "capstone9_smartmeter"
schema_name = "smart_grid_bronze"

base_path = "/Volumes/capstone9_smartmeter/smart_grid_bronze/raw_volume"

# Set the session's default catalog and schema
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("Catalog:", catalog_name)
print("Schema:", schema_name)
print("Volume:", base_path)

In [0]:
base_path = "/Volumes/capstone9_smartmeter/smart_grid_bronze/raw_volume"

schema_base = f"{base_path}/_schemas/smart_grid_new"

checkpoint_base = f"{base_path}/_checkpoints/smart_grid_new"

In [0]:
schema_folders = [
    "readings",
    "outages",
    "recovery"
]

for folder in schema_folders:
    dbutils.fs.mkdirs(f"{schema_base}/{folder}")

print("Schema folders created successfully")

In [0]:
checkpoint_folders = [
    "readings",
    "outages",
    "recovery"
]

for folder in checkpoint_folders:
    dbutils.fs.mkdirs(f"{checkpoint_base}/{folder}")

print("Checkpoint folders created successfully")

In [0]:
print("SCHEMA FOLDERS")
display(dbutils.fs.ls(schema_base))

print("CHECKPOINT FOLDERS")
display(dbutils.fs.ls(checkpoint_base))

In [0]:
display(dbutils.fs.ls(base_path))

In [0]:
folders = [
    "telemetry_data",
    "event_data",
    "master_data",
    "lookup_data"
]

for folder in folders:

    print(f"\n========== {folder} ==========")

    files = dbutils.fs.ls(f"{base_path}/{folder}")

    for file in files:
        print(file.name, file.path)

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [0]:
# Count all records in CSVs in raw_volume (telemetry_data has subdirectories)
total_records = 0
for folder in ["telemetry_data", "event_data", "master_data", "lookup_data"]:
    path = f"{base_path}/{folder}"
    top_items = dbutils.fs.ls(path)
    csv_files = []
    for item in top_items:
        if item.name.endswith("/"):
            # It's a subdirectory — list its contents for CSVs
            sub_files = dbutils.fs.ls(item.path)
            csv_files.extend([f.path for f in sub_files if f.name.endswith(".csv")])
        elif item.name.endswith(".csv"):
            csv_files.append(item.path)
    if csv_files:
        folder_count = spark.read.option("header", "true").csv(csv_files).count()
    else:
        folder_count = 0
    total_records += folder_count
    print(f"{folder}: {len(csv_files)} CSV files, {folder_count:,} records")

print(f"\nTotal records across all CSVs: {total_records:,}")

AUTOLOADER


In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
readings_schema = StructType([
    StructField("reading_id", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("substation_id", StringType(), True),
    StructField("interval_start_ts", TimestampType(), True),
    StructField("interval_end_ts", TimestampType(), True),
    StructField("energy_kwh", DoubleType(), True),
    StructField("voltage_v", DoubleType(), True),
    StructField("current_a", DoubleType(), True),
    StructField("power_factor", DoubleType(), True),
    StructField("quality_code", StringType(), True),
    StructField("reading_source", StringType(), True),
    StructField("received_ts", TimestampType(), True)
])

print("Readings schema created")
print(readings_schema.simpleString())

outage_schema = StructType([
    StructField("outage_id", StringType(), True),
    StructField("substation_id", StringType(), True),
    StructField("outage_start_ts", TimestampType(), True),
    StructField("outage_end_ts", TimestampType(), True),
    StructField("outage_cause", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("affected_meters", IntegerType(), True),
    StructField("incident_status", StringType(), True),
    StructField("reported_ts", TimestampType(), True),
    StructField("repair_team", StringType(), True)
])

print("Outage schema created")
print(outage_schema.simpleString())


recovery_schema = StructType([
    StructField("recovery_event_id", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("outage_id", StringType(), True),
    StructField("event_ts", TimestampType(), True),
    StructField("recovery_type", StringType(), True),
    StructField("signal_strength", DoubleType(), True),
    StructField("firmware_version", StringType(), True),
    StructField("acknowledged_flag", BooleanType(), True),
    StructField("technician_id", StringType(), True),
    StructField("received_ts", TimestampType(), True)
])

print("Recovery schema created")
print(recovery_schema.simpleString())

In [0]:
base_path = "/Volumes/capstone9_smartmeter/smart_grid_bronze/raw_volume"

display(dbutils.fs.ls(f"{base_path}/master_data"))
display(dbutils.fs.ls(f"{base_path}/lookup_data"))

In [0]:
from pyspark.sql import functions as F

readings_path = f"{base_path}/telemetry_data"

readings_schema = f"{schema_base}/readings"

readings_checkpoint = f"{checkpoint_base}/readings"

readings_table = f"{catalog_name}.{schema_name}.bronze_smart_meter_readings"